# Lung Nematic — Colab analysis

Graphical front-end over the same analysis engine as the command line:
`analyze_folder` → `analyze_image`.

This version preserves recursive folder structure, accepts a metadata CSV,
keeps errors from every requested field, and writes both combined and
per-field group summaries. A folder layout such as
`images/control/*.tif` and `images/fibrosis/*.tif` therefore retains the
group names instead of flattening every image into one directory.

Run the cells from top to bottom. For a large study, store source images in
Drive and leave results under `/content` while processing; download the ZIP
at the end.

In [ ]:
#@title 1 · Install the current repository version { display-mode: "form" }
import subprocess
import sys

repository = "https://github.com/Danpc11/lung-nematic.git\nbranch = "main\npackage_url = f"git+{repository}@{branch}"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", package_url],
    check=True,
)

# Colab retains imported modules between cell executions. Purge a previous
# copy so rerunning setup really loads the newly installed commit.
for name in [
    module for module in list(sys.modules)
    if module == "lung_nematic" or module.startswith("lung_nematic.")
]:
    del sys.modules[name]

import lung_nematic
from importlib.metadata import version

print(f"Ready: lung-nematic {version('lung-nematic')}")
print(f"Loaded from: {lung_nematic.__file__}")

In [ ]:
#@title 2 · Load images and optional metadata { display-mode: "form" }
import io
import os
import shutil
import zipfile
from collections import Counter
from pathlib import Path

image_source = "Google Drive folder"  #@param ["Google Drive folder", "Upload images or ZIP"]
drive_folder = "MyDrive/lung_histology"  #@param {type:"string"}
clear_previous_uploads = True  #@param {type:"boolean"}

metadata_source = "None"  #@param ["None", "Upload metadata CSV", "CSV path"]
metadata_csv_path = "metadata.csv"  #@param {type:"string"}

SUPPORTED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"
}

if image_source == "Google Drive folder":
    from google.colab import drive

    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    candidate = Path(drive_folder)
    IMAGES_DIR = (
        candidate if candidate.is_absolute()
        else Path("/content/drive") / candidate
    )
    if not IMAGES_DIR.is_dir():
        raise FileNotFoundError(f"Drive folder not found: {IMAGES_DIR}")
else:
    from google.colab import files

    IMAGES_DIR = Path("/content/images")
    if clear_previous_uploads and IMAGES_DIR.exists():
        shutil.rmtree(IMAGES_DIR)
    IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No files were uploaded.")

    root = IMAGES_DIR.resolve()
    for filename, contents in uploaded.items():
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(contents)) as archive:
                for member in archive.infolist():
                    destination = (IMAGES_DIR / member.filename).resolve()
                    if destination != root and root not in destination.parents:
                        raise ValueError(
                            f"Unsafe path in ZIP archive: {member.filename}"
                        )
                    if member.is_dir():
                        destination.mkdir(parents=True, exist_ok=True)
                    else:
                        destination.parent.mkdir(parents=True, exist_ok=True)
                        with archive.open(member) as source, destination.open("wb") as target:
                            shutil.copyfileobj(source, target)
        elif Path(filename).suffix.lower() in SUPPORTED_EXTENSIONS:
            destination = IMAGES_DIR / Path(filename).name
            if destination.exists():
                raise FileExistsError(
                    f"Upload would overwrite an existing image: {destination.name}"
                )
            destination.write_bytes(contents)
        elif filename.lower().endswith(".csv") and metadata_source == "None":
            print(
                f"Ignoring {filename} as metadata_source is None. "
                "Select Upload metadata CSV to use it."
            )
        else:
            print(f"Ignoring unsupported upload: {filename}")

METADATA_CSV = None
if metadata_source == "Upload metadata CSV":
    from google.colab import files

    uploaded_metadata = files.upload()
    csv_files = [
        (name, contents) for name, contents in uploaded_metadata.items()
        if name.lower().endswith(".csv")
    ]
    if len(csv_files) != 1:
        raise ValueError(
            f"Upload exactly one metadata CSV; received {len(csv_files)}."
        )
    METADATA_CSV = Path("/content/metadata.csv")
    METADATA_CSV.write_bytes(csv_files[0][1])
elif metadata_source == "CSV path":
    candidate = Path(metadata_csv_path)
    METADATA_CSV = candidate if candidate.is_absolute() else IMAGES_DIR / candidate
    if not METADATA_CSV.is_file():
        raise FileNotFoundError(f"Metadata CSV not found: {METADATA_CSV}")

from lung_nematic.io_utils import discover_images

image_paths = discover_images(IMAGES_DIR)
if not image_paths:
    raise FileNotFoundError(
        f"No supported images were found recursively under {IMAGES_DIR}."
    )

stems = Counter(path.stem for path in image_paths)
duplicate_stems = sorted(stem for stem, count in stems.items() if count > 1)
print(f"Image root: {IMAGES_DIR}")
print(f"Metadata: {METADATA_CSV or 'none'}")
print(f"{len(image_paths)} supported image(s):")
for path in image_paths[:30]:
    print("  ", path.relative_to(IMAGES_DIR))
if len(image_paths) > 30:
    print(f"  ... and {len(image_paths) - 30} more")
if duplicate_stems and METADATA_CSV is None:
    print()
    print("Duplicate filename stems detected:", duplicate_stems)
    print(
        "Provide metadata with relative_path and unique image_id values; "
        "otherwise the batch driver will stop to prevent overwritten results."
    )

In [ ]:
#@title 2b · Calibrate the director field { display-mode: "form" }
#@markdown Tune the orientation field the way OrientationJ is tuned: pick an
#@markdown image, choose the field, and adjust the integration scale (σ), the
#@markdown vector grid, and the gradient scale until the vectors follow the
#@markdown structures you see. The values you settle on go into cell 3.
#@markdown
#@markdown Bigger **σ** = smoother, longer-range orientation. Bigger **grid** =
#@markdown fewer, larger vectors. **Gradient scale** suppresses pixel noise
#@markdown before the derivative; keep it small (1–2 px).

image_to_preview = 0  #@param {type:"integer"}
field_type = "collagen (eosin texture)"  #@param ["collagen (eosin texture)", "nuclear (segmented nuclei)"]
integration_sigma_px = 20  #@param {type:"slider", min:6, max:60, step:2}
vector_grid_px = 18  #@param {type:"slider", min:8, max:48, step:2}
gradient_scale_px = 1.5  #@param {type:"slider", min:0.5, max:4, step:0.5}
vector_linewidth = 1.0  #@param {type:"slider", min:0.4, max:2.0, step:0.1}
min_order_to_draw = 0.03  #@param {type:"slider", min:0, max:0.3, step:0.01}

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from lung_nematic.preprocessing import make_tissue_mask
from lung_nematic.collagen_field import compute_collagen_field
from lung_nematic.visualization import draw_dense_director

path = image_paths[image_to_preview]
rgb = np.array(Image.open(path).convert("RGB"))
gray = np.array(Image.open(path).convert("L"))
mask, hed = make_tissue_mask(rgb)

if field_type.startswith("collagen"):
    field = compute_collagen_field(
        hed[:, :, 1], sigma_px=float(integration_sigma_px),
        inner_scale_px=float(gradient_scale_px),
        tissue_mask=mask, mask_normalized=True,
    )
else:
    # nuclear route: orientation from the hematoxylin structure tensor, which
    # tracks elongated nuclei the same way the eosin tensor tracks fibres
    field = compute_collagen_field(
        hed[:, :, 0], sigma_px=float(integration_sigma_px),
        inner_scale_px=float(gradient_scale_px),
        tissue_mask=mask, mask_normalized=True,
    )

n_vectors = draw_dense_director(
    gray, field, mask, "/content/calibration_preview.png",
    grid_step_px=int(vector_grid_px), min_order=float(min_order_to_draw),
    linewidth=float(vector_linewidth), color="yellow",
    title=f"{field_type.split()[0]}  σ={integration_sigma_px}  grid={vector_grid_px}",
)

from IPython.display import Image as _Image, display
display(_Image("/content/calibration_preview.png"))

global_S = float(np.sqrt(
    np.mean(np.cos(2 * field["theta"][mask])) ** 2
    + np.mean(np.sin(2 * field["theta"][mask])) ** 2
))
print(f"{n_vectors} vectors drawn   global order S = {global_S:.3f}   "
      f"median local S = {np.median(field['order'][mask]):.3f}")
print()
print("When the field looks right, copy these into cell 3:")
print(f"  sigmas_px around {integration_sigma_px}   "
      f"defect_grid_step_px = {vector_grid_px}   "
      f"collagen_inner_scale_px = {gradient_scale_px}")

In [ ]:
#@title 3 · Analysis parameters { display-mode: "form", run: "auto" }
fields = "collagen"  #@param ["nuclear", "collagen", "fused", "nuclear + collagen", "all three"]
run_null = True  #@param {type:"boolean"}
run_colocalization = True  #@param {type:"boolean"}

sigmas_px = "40, 55, 70, 85"  #@param {type:"string"}
defect_grid_step_px = 24  #@param {type:"slider", min:8, max:64, step:4}
min_edge_distance_px = 30  #@param {type:"slider", min:0, max:120, step:5}
density_quantile = 0.45  #@param {type:"slider", min:0, max:0.9, step:0.05}
min_scales_for_persistence = 2  #@param {type:"slider", min:1, max:4, step:1}
defect_cluster_radius_px = 70  #@param {type:"slider", min:20, max:150, step:5}

detect_integer_defects = False  #@param {type:"boolean"}
integer_defect_loop_radius_px = 30  #@param {type:"slider", min:8, max:80, step:2}
integer_defect_loop_points = 8  #@param {type:"slider", min:6, max:24, step:1}
integer_min_separation_px = 60  #@param {type:"slider", min:0, max:200, step:5}

save_diagnostic_panel = True  #@param {type:"boolean"}
save_defect_maps = False  #@param {type:"boolean"}
defect_map_window_px = 220  #@param {type:"slider", min:80, max:400, step:20}
mask_normalized_smoothing = False  #@param {type:"boolean"}
collagen_inner_scale_px = 1.5  #@param {type:"number"}

n_permutations = 199  #@param {type:"slider", min:2, max:1000, step:1}
null_downsample = 2  #@param {type:"slider", min:1, max:6, step:1}
null_n_jobs = -1  #@param {type:"integer"}
null_mode = "shuffle"  #@param ["shuffle", "uniform"]
n_bootstrap = 2000  #@param {type:"slider", min:2, max:5000, step:1}

microns_per_pixel = 0.0  #@param {type:"number"}
random_seed = 42  #@param {type:"integer"}

from lung_nematic.config import AnalysisConfig

parsed_sigmas = tuple(
    float(value.strip()) for value in sigmas_px.split(",") if value.strip()
)
if not parsed_sigmas:
    raise ValueError("sigmas_px must contain at least one value.")

config = AnalysisConfig(
    sigmas_px=parsed_sigmas,
    defect_grid_step_px=int(defect_grid_step_px),
    min_edge_distance_px=int(min_edge_distance_px),
    density_quantile=float(density_quantile),
    min_scales_for_persistence=min(
        int(min_scales_for_persistence), len(parsed_sigmas)
    ),
    defect_cluster_radius_px=float(defect_cluster_radius_px),
    detect_integer_defects=bool(detect_integer_defects),
    integer_defect_loop_radius_px=int(integer_defect_loop_radius_px),
    integer_defect_loop_points=int(integer_defect_loop_points),
    integer_min_separation_px=float(integer_min_separation_px),
    save_diagnostic_panel=bool(save_diagnostic_panel),
    save_defect_maps=bool(save_defect_maps),
    defect_map_window_px=int(defect_map_window_px),
    mask_normalized_smoothing=bool(mask_normalized_smoothing),
    collagen_inner_scale_px=float(collagen_inner_scale_px),
    field_type="nuclear",
    run_null=bool(run_null),
    run_colocalization=bool(run_colocalization),
    n_permutations=int(n_permutations),
    null_downsample=int(null_downsample),
    null_n_jobs=int(null_n_jobs),
    null_mode=null_mode,
    n_bootstrap=int(n_bootstrap),
    default_microns_per_pixel=(float(microns_per_pixel) or None),
    random_seed=int(random_seed),
)
config.validate()

FIELD_SELECTIONS = {
    "nuclear": ["nuclear"],
    "collagen": ["collagen"],
    "fused": ["fused"],
    "nuclear + collagen": ["nuclear", "collagen"],
    "all three": ["nuclear", "collagen", "fused"],
}
FIELDS = FIELD_SELECTIONS[fields]
print(
    f"Config ready: fields={FIELDS}, null={config.run_null}, "
    f"colocalization={config.run_colocalization}"
)

In [ ]:
#@title 4 · Run analysis { display-mode: "form" }
import shutil
import time
from dataclasses import replace
from pathlib import Path

import pandas as pd
from lung_nematic.batch import analyze_folder

clear_previous_results = True  #@param {type:"boolean"}
stop_on_first_error = False  #@param {type:"boolean"}
RESULTS_DIR = Path("/content/results")

# Clearing once before the field loop prevents stale overlays, media, and
# error reports from a previous run without erasing fields generated in
# the current run.
if clear_previous_results and RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

n_images = len(image_paths)
import os

null_workers = (
    os.cpu_count() or 1
    if config.null_n_jobs == -1
    else config.null_n_jobs
)
per_permutation_seconds = 6.0 / max(1.0, (config.null_downsample / 2.0) ** 2)
per_image_seconds = 70.0
if config.run_null:
    per_image_seconds += (
        per_permutation_seconds * config.n_permutations / max(1, null_workers)
    )
if config.run_colocalization:
    per_image_seconds += 13.0 * (config.n_bootstrap / 2000.0)
estimated_minutes = (
    per_image_seconds * n_images * len(FIELDS) / 60.0
)
print(
    f"{n_images} image(s) × {len(FIELDS)} field(s). "
    f"Heuristic estimate: ~{estimated_minutes:.0f} min total."
)
if estimated_minutes > 20:
    print(
        "Long run: turn off the null model, lower n_permutations, or "
        "increase null_downsample or null_n_jobs when appropriate."
    )

summaries_by_field = []
errors_by_field = []
started = time.time()
for field_name in FIELDS:
    print(f"\n=== field: {field_name} ===")
    field_config = replace(config, field_type=field_name)
    field_summary, field_errors = analyze_folder(
        IMAGES_DIR,
        RESULTS_DIR,
        config=field_config,
        metadata_csv=METADATA_CSV,
        continue_on_error=not stop_on_first_error,
    )
    if not field_summary.empty:
        field_summary["field_type"] = field_name
        summaries_by_field.append(field_summary)
    if not field_errors.empty:
        field_errors["field_type"] = field_name
        errors_by_field.append(field_errors)
        display(field_errors)
    print(f"processed {len(field_summary)}, failed {len(field_errors)}")

summary = (
    pd.concat(summaries_by_field, ignore_index=True)
    if summaries_by_field else pd.DataFrame()
)
processing_errors = (
    pd.concat(errors_by_field, ignore_index=True)
    if errors_by_field else pd.DataFrame()
)

# Rebuild final batch-level files after every field has run. Individual
# analyze_folder calls write temporary per-field versions of these files.
summary.to_csv(RESULTS_DIR / "summary_metrics.csv", index=False)
errors_path = RESULTS_DIR / "processing_errors.csv"
if processing_errors.empty:
    if errors_path.exists():
        errors_path.unlink()
else:
    processing_errors.to_csv(errors_path, index=False)

if not summary.empty:
    metric_candidates = [
        "global_nematic_order_S",
        "local_S_median",
        "n_defects_total",
        "n_plus_half",
        "n_minus_half",
        "n_plus_one",
        "n_minus_one",
        "net_topological_charge",
        "defect_density_mm2",
        "mean_defect_confidence",
    ]
    metrics = [name for name in metric_candidates if name in summary.columns]
    group_summary = (
        summary.groupby(["field_type", "group"])[metrics]
        .agg(["count", "mean", "median", "std"])
    )
    group_summary.to_csv(RESULTS_DIR / "group_summary.csv")
    for field_name, field_frame in summary.groupby("field_type"):
        field_metrics = [name for name in metrics if name in field_frame.columns]
        (
            field_frame.groupby("group")[field_metrics]
            .agg(["count", "mean", "median", "std"])
            .to_csv(RESULTS_DIR / f"group_summary_{field_name}.csv")
        )

elapsed_minutes = (time.time() - started) / 60.0
print(
    f"\nFinished in {elapsed_minutes:.1f} min: "
    f"{len(summary)} successful rows, {len(processing_errors)} failures."
)
if not summary.empty:
    display(summary)
if not processing_errors.empty:
    print("Review processing_errors.csv before interpreting group summaries.")

In [ ]:
#@title 5 · Inspect results { display-mode: "form" }
from IPython.display import Image as IPyImage, display

max_overlays_to_show = 30  #@param {type:"slider", min:1, max:100, step:1}
max_defect_maps_per_image = 12  #@param {type:"slider", min:0, max:50, step:1}

overlays = sorted(RESULTS_DIR.glob("*/*_overlay.png"))
if not overlays:
    print("No overlays found. Run the analysis cell and inspect any failures.")
else:
    print(f"Showing {min(len(overlays), max_overlays_to_show)} of {len(overlays)} overlays")
    for overlay in overlays[:max_overlays_to_show]:
        print("\n", overlay.relative_to(RESULTS_DIR))
        display(IPyImage(filename=str(overlay), width=880))

        tag = overlay.name.removesuffix("_overlay.png")
        histogram = overlay.parent / f"{tag}_null_hist.png"
        if histogram.exists():
            display(IPyImage(filename=str(histogram), width=520))
        defect_maps = sorted(
            (overlay.parent / "defect_maps").glob(f"{tag}_defect*.png")
        )
        for defect_map in defect_maps[:max_defect_maps_per_image]:
            display(IPyImage(filename=str(defect_map), width=420))

summary_path = RESULTS_DIR / "summary_metrics.csv"
if summary_path.exists() and summary_path.stat().st_size > 1:
    print("\nCombined summary:")
    display(pd.read_csv(summary_path))
if (RESULTS_DIR / "processing_errors.csv").exists():
    print("\nProcessing errors:")
    display(pd.read_csv(RESULTS_DIR / "processing_errors.csv"))

In [ ]:
#@title 6 · Download all results { display-mode: "form" }
import shutil

if not RESULTS_DIR.exists() or not any(RESULTS_DIR.iterdir()):
    raise FileNotFoundError("No results are available to download.")

archive = shutil.make_archive(
    "/content/lung_nematic_results",
    "zip",
    root_dir=RESULTS_DIR,
)
print(f"Archive ready: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")
from google.colab import files

files.download(archive)

## Output integrity notes

- `summary_metrics.csv` contains every successful image-field combination.
- `processing_errors.csv` contains failures from **all** requested fields.
- `group_summary.csv` groups by both field and biological group.
- `group_summary_<field>.csv` provides a simpler table for each field.
- When filenames repeat across folders, metadata should use
  `relative_path` and a unique `image_id` to prevent output collisions.
- Rerunning analysis with `clear_previous_results=True` ensures the ZIP
  cannot contain stale files from an earlier parameter set.